# DL4CV

<h3> Learning objectives </h3>

After this session you should ...
<list>
    <li> ... understand how image segmentation performed </li>
    <li> ... be able to explain the difference between FCN and U-Net </li>
    <li> ... understand the current challenges in image segmentation </li>
</list>

<h3> Image Segmentation </h3>

Is the assinment of a class label to each pixel <br>
Therefore specific labeled training data needs to exists <br>
The image might have n channels for n different labels <br>


![alt text](https://paperswithcode.com/media/thumbnails/task/task-0000000885-88f43a92.jpg)
Source: Vasilev, 2019

<h3> Encoder Decoder and FCN </h3>

To train an image segmentation model, a specific type of network is used <br>
FCN stands for Fully convolutional network and has no dense layers. <br>

It is an encoder-decoder network.<br>
The encoder transform the image to to an highly abstract representation of the image.. <br>
The decoder uses this abstract representation to transform the image into the labeled ground truth data <br>
Source: Vasilev, 2019

Image segmentation a special application in which the image is seperated and segmented in different objects. Therefore special training data is necessary, the normal image and the output which is also an image. However, this image contains n channels for n different classes, and for each pixel, the class affiliation is predicted. Image segmentation is necessary for different taks like autonomous driving to separate street from sidewalk or cars, which can not be done by classification or object detection. Neural networks for image segmentation are special as they are encoder-decoder networks as the input is an image, and the output is also an image. The encoder part of the image transforms the original image into a higher-order representation. The decoder uses the higher-order representation to build the output image. The networks used are fully convolutional networks existing only of convolution and pooling layers and NO fully connected/dense layers (Vasilev, 2019,  Long, Shelhamer & Darrell, 2015). 

In [ ]:
!nvidia-smi # check if GPU is available

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import layers, models, optimizers, metrics
from tensorflow.keras import backend as K 
import cv2
import os
from tqdm import tqdm
from matplotlib import pyplot as plt
import gc

In [ ]:
# Download data 
# WARNING! DONWLOADSIZE of 5.11 G
import os
os.environ['KAGGLE_USERNAME'] = "" # username from the json file
os.environ['KAGGLE_KEY'] = "" # key from the json file
! kaggle datasets download kumaresanmanickavelu/lyft-udacity-challenge

In [ ]:
os.system("unzip lyft-udacity-challenge.zip") # unzip data

In [ ]:
def read_data(name_set:str):
    X=[]
    y=[]
    
    for name in name_set:
        path="./" + name +"/" + name
        image_list = os.listdir(path+"/CameraRGB") # read file with all image names
        
        for i in tqdm(range(0, len(image_list))): # for all images import original image and mask
            image_name_org = path+"/CameraRGB"+"/"+image_list[i]
            image_name_mask = path+"/CameraSeg"+"/"+image_list[i]
            org_img = cv2.imread(image_name_org)
            mask_img = cv2.imread(image_name_mask)

            if (org_img.all() and mask_img.all()) is not None: # resize original image and mask
                resize_org = cv2.resize(org_img, (256,256))
                resize_mask = cv2.resize(mask_img, (256,256))
                X.append(resize_org) # append to output
                y.append(resize_mask)
    return X,y 

In [ ]:
X, y = read_data(name_set=["dataA", "dataB"]) # read data from dataset partition A and B
gc.collect() # clear uncessary RAM

In [ ]:
# Transform data to numpy n-dimensional array
X=np.array(X)
y=np.array(y)

In [ ]:
# Inspect shapes 
print(X.shape)
print(y.shape) # It seems after reading the output that it has only 3 channels; However, this is not true; The classes are encoded as integers in the last channel

In [ ]:
# Data inspection for the second observation

temp = X[1]
temp_mask = y[1]
mask = np.array([max(temp_mask[i, j]) for i in range(temp_mask.shape[0]) for j in range(temp_mask.shape[1])]).reshape(temp.shape[0], temp.shape[1]) # get the maximum pixel value for each pixel
plt.imshow(temp)
plt.show()

plt.imshow(mask, cmap="Paired")
plt.show()

In [ ]:
# inspect number of classes 
print(temp_mask[:,:,2].min())
print(temp_mask[:,:,2].max()) # 13 classes

In [ ]:
num_classes=13

In [ ]:
def transform_mask(y): # write function to transform the target variables to masked arrays with one channel per class
    result = []
    for i in tqdm(range(0, len(y))):
        temp=y[i]
        temp = temp[:,:,2] # get the data and the stored information class in the third channel (values from 0 to 12)
        transformed_image = np.zeros(shape=(temp.shape[0], temp.shape[1], num_classes), dtype=np.uint8) # create empty image
        for j in range(0, num_classes): # for each class
            zeros = np.zeros(shape=(temp.shape[0], temp.shape[1]), dtype=np.uint8)
            zeros[np.where(temp==j)[0], np.where(temp==j)[1]]=1 # where the class exisits create 1 else zeros for that channel
            transformed_image[:,:,j] = zeros # overwrite channel in the empty image
        result.append(transformed_image)
    return np.array(result)

In [ ]:
transformed_y = transform_mask(y=y) # transform targets
gc.collect()

In [ ]:
print(transformed_y.shape) # inspect chapes 

In [ ]:
print(transformed_y[0][:,:,0]) # Inspect the first obersvation with the zero's class

<h3> FCN </h3>
<list>
    <li>by Long, Shelhamer & Darrell, 2015 </li>
    <li>first suggestion of a fully connected network </li>
    <li>No dense layers in the network </li>
    <li>Encoder: Classical CNN e.g. VGG </li>
    <li>Decoder: 1 by 1 Convolutions + Upsampling </li>
    <li>Different versions: FCN-32, FCN-16, FCN-8 </li>
</list>

![alt text](http://deeplearning.net/tutorial/_images/cat_segmentation.png)
Source:  Long, Shelhamer & Darrell, 2015

<h3> FCN </h3>
<list>
    <li>Number counts the upsampling in the last step</li>
    <li>Different "branches" exists, added up</li>
</list>

![alt text](https://www.researchgate.net/publication/327521314/figure/fig1/AS:668413361930241@1536373572028/Fully-convolutional-neural-network-architecture-FCN-8.ppm)

Image Source:  https://www.researchgate.net/figure/Fully-convolutional-neural-network-architecture-FCN-8_fig1_327521314

The first algorithm applied for segmentation is thus called "fully convolutional network" and was published in 2015. The encoder is a normal CNN e.g. VGG. The decoder are 1 by 1 convolutions plus an upsampling layer. Different versions of the FCN exist, with some connecting layers of the encoder and decoder. In the original paper, a deconvolutional layer is used for the upsampling. However, we used here normal upsampling layer with bilinear interpolation. This technique can upsample the input by just duplicating the rows and columns ( Long, Shelhamer & Darrell, 2015).

In [ ]:
# FCN-32

def FCN():
    input_ = layers.Input(shape=(256,256,3))
    
    conv = layers.Conv2D(32,3, activation="relu", padding="same") (input_)
    conv = layers.MaxPooling2D((2,2)) (conv)
    
    conv = layers.Conv2D(64,3, activation="relu", padding="same") (conv)
    conv = layers.MaxPooling2D((2,2)) (conv)
    
    conv = layers.Conv2D(128,3, activation="relu", padding="same") (conv)
    conv = layers.MaxPooling2D((2,2)) (conv)
    
    conv = layers.Conv2D(256,3, activation="relu", padding="same") (conv)
    conv = layers.MaxPooling2D((2,2)) (conv)
    
    conv = layers.Conv2D(512,3, activation="relu", padding="same") (conv)
    conv = layers.MaxPooling2D((2,2)) (conv)
    
    conv = layers.Conv2D(4096,1, activation="relu", padding="same") (conv)
    
    conv = layers.Conv2D(num_classes,1, activation="softmax", padding="same") (conv)
    
    upsampling = layers.UpSampling2D((32,32)) (conv)
    m = models.Model(input_,upsampling)
    
    m.compile(loss="categorical_crossentropy", optimizer=optimizers.Adam(lr=0.0001), metrics=["accuracy", metrics.MeanIoU(num_classes=13)])
    return m

In [ ]:
model = FCN()

In [ ]:
print(model.summary())

In [ ]:
model.fit(X, transformed_y, batch_size=16, epochs=25, validation_split=0.1) # train fcn-32 model

In [ ]:
# Show the true value of Y 

mask = np.argmax(transformed_y[1],axis=2)
print(mask.shape)

plt.imshow(mask, cmap="Paired")
plt.show()

In [ ]:
print(mask)

In [ ]:
# Show a sample prediction for the second observation of the train set
sample_img = X[1]
sample_img = sample_img[None, :]
prediction = model.predict(np.float32(sample_img))
prediction = prediction[0]
prediction_mask = np.argmax(prediction, axis=2) # backtransform prediction from multi-channel to one channel
print(prediction_mask)

In [ ]:
# Show prediction
plt.imshow(prediction_mask, cmap="Paired")
plt.show()

In [ ]:
# Save model and clear session and empty RAM to save resources 
model.save('fcn.h5')
del model
K.clear_session()
gc.collect()

<h3>Hands on ... </h3>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fc/Ic_assignment_48px.svg/1024px-Ic_assignment_48px.svg.png" width=50 height=50 align="left">
Build a FCN-16 with multiple branches based on the network architecture used above

In [ ]:
#### Your code goes here #########################

def FCN_16():
    input_ = layers.Input(shape=(256,256,3))
    
    conv_1 = layers.Conv2D(32,3, activation="relu", padding="same") (input_)
    conv_1 = layers.MaxPooling2D((2,2)) (conv_1)
    
    conv_2 = layers.Conv2D(64,3, activation="relu", padding="same") (conv_1)
    conv_2 = layers.MaxPooling2D((2,2)) (conv_2)
    
    conv_3 = layers.Conv2D(128,3, activation="relu", padding="same") (conv_2)
    conv_3 = layers.MaxPooling2D((2,2)) (conv_3)
    
    conv_4 = layers.Conv2D(256,3, activation="relu", padding="same") (conv_3)
    conv_4 = layers.MaxPooling2D((2,2)) (conv_4)

    conv_4_pred = layers.Conv2D(num_classes,1, activation="relu", padding="same") (conv_4)

    conv_5 = layers.Conv2D(512,3, activation="relu", padding="same") (conv_4)
    conv_5 = layers.MaxPooling2D((2,2)) (conv_5)
    
    
    
    conv_5 = layers.Conv2D(4096,1, activation="relu", padding="same") (conv_5)
    
    
    pred = layers.Conv2D(num_classes,1, activation="relu", padding="same") (conv_5)
    pred = layers.UpSampling2D((2,2)) (pred)

    add = layers.Add()([conv_4_pred, pred])

    conv = layers.Conv2D(num_classes,1, activation="softmax", padding="same") (add) #filter size
    
    upsampling = layers.UpSampling2D((16,16)) (conv) # replace by deconvolution
    m = models.Model(input_,upsampling)
    
    m.compile(loss="categorical_crossentropy", optimizer=optimizers.Adam(lr=0.001), metrics=["accuracy", metrics.MeanIoU(num_classes=13)])
    return m

In [ ]:
fcn16=FCN_16()
print(fcn16.summary())

In [ ]:
fcn16.fit(X, transformed_y, batch_size=32, epochs=25, validation_split=0.1) # Train fcn8 model 

In [ ]:
# Show sample prediction for fcn 8
mask = np.argmax(transformed_y[1],axis=2)
print(mask.shape)
plt.imshow(mask, cmap="Paired")
plt.show() # original image 

sample_img = X[1]
sample_img = sample_img[None, :]
prediction = fcn16.predict(np.float32(sample_img))
prediction = prediction[0]
prediction_mask = np.argmax(prediction, axis=2)
plt.imshow(prediction_mask, cmap="Paired")
plt.show() # prediction

In [ ]:
# Save model and clear session and empty RAM to save resources
fcn16.save("fcn16.h5")
del fcn16
K.clear_session()
gc.collect()

<h3>U-Net</h3>
<list>
    <li>by Ronneberger, Fischer & Brox, 2015 </li>
    <li>Uses also the features from lower level layers</li>
    <li>Performs upsampling by deconvolution </li>
</list>

![alt text](https://lmb.informatik.uni-freiburg.de/people/ronneber/u-net/u-net-architecture.png)

Source: Ronneberger, Fischer & Brox, 2015

<h3>Deconvolution </h3>
<list>
    <li>Transposed Convolution, Deconvolution, sub-pixel convolution</li>
    <li>"Operation to create an image out of an convoluted input" </li>
    
</list>

![alt text](https://miro.medium.com/max/1400/1*kOThnLR8Fge_AJcHrkR3dg.gif)

Image source: Lane, 2018
Source: Dumoulin & Visin, 2016

For the other often-used network, the U-Net, used in medicine, so-called deconvolutions are used. Deconvolution, also known as transposed convolution or subpixel convolution, can transform the input through applying a kernel so that the output has more dimensions. The best and most intuitive way of thinking of a deconvolution is depicted above. In this case, a single input value is multiplied by a kernel, the black square box with dimensions 3 by 3 to build a new output of 3 by 3. With strides 1, the blue input of 4 by 4 is transformed to an 6 by 6 output (Ronneberger, Fischer & Brox, 2015, Dumoulin & Visin, 2016). 

In [ ]:
def unet():
    # https://www.depends-on-the-definition.com/unet-keras-segmenting-images/
    input_ = layers.Input(shape=(256,256,3))
    conv_1 = layers.Conv2D(32,3, activation="relu", padding="same") (input_)
    pool1 = layers.MaxPooling2D((2,2)) (conv_1)
    
    conv_2 = layers.Conv2D(64,3, activation="relu", padding="same") (pool1)
    pool2 = layers.MaxPooling2D((2,2)) (conv_2)
    
    conv_3 = layers.Conv2D(128,3, activation="relu", padding="same") (pool2)
    pool3 = layers.MaxPooling2D((2,2)) (conv_3)
    
    conv_4 = layers.Conv2D(128,3, activation="relu", padding="same") (pool3)
    pool4 = layers.MaxPooling2D((2,2)) (conv_4)
    
    conv_5 = layers.Conv2D(256,3, activation="relu", padding="same") (pool4)
    pool5 = layers.MaxPooling2D((2,2)) (conv_5)
    
    deconv_1 = layers.Conv2DTranspose(256,3,strides=(2,2), activation="relu", padding="same") (pool5)
    add_1 = layers.Concatenate()([deconv_1, conv_5])
    deconv_1_conv =layers.Conv2D(256,3, activation="relu", padding="same") (add_1)
    
    deconv_2 = layers.Conv2DTranspose(128,3,strides=(2,2), activation="relu", padding="same") (deconv_1_conv)
    add_2 = layers.Concatenate()([deconv_2, conv_4])
    deconv_2_conv =layers.Conv2D(128,3, activation="relu", padding="same") (add_2)
    
    deconv_3 = layers.Conv2DTranspose(128,3,strides=(2,2), activation="relu", padding="same") (deconv_2_conv)
    add_3 = layers.Concatenate()([deconv_3, conv_3])
    deconv_3_conv =layers.Conv2D(128,3, activation="relu", padding="same") (add_3)
    
    deconv_4 = layers.Conv2DTranspose(64,3,strides=(2,2), activation="relu", padding="same") (deconv_3_conv)
    add_4 = layers.Concatenate()([deconv_4, conv_2])
    deconv_4_conv =layers.Conv2D(64,3, activation="relu", padding="same") (add_4)
    
    deconv_5 = layers.Conv2DTranspose(32,3,strides=(2,2), activation="relu", padding="same") (deconv_4_conv)
    add_5 = layers.Concatenate()([deconv_5, conv_1])
    deconv_5_conv =layers.Conv2D(32,3, activation="relu", padding="same") (add_5)
    
    output = layers.Conv2D(num_classes, 1, activation="softmax", padding="same") (deconv_5_conv)
    m = models.Model(input_,output)
    m.compile(loss="categorical_crossentropy", optimizer=optimizers.Adam(lr=0.001), metrics=["accuracy", metrics.MeanIoU(num_classes=13)])
    return m

In [ ]:
unet_model = unet()

In [ ]:
print(unet_model.summary())

In [ ]:
unet_model.fit(X, transformed_y, batch_size=16, epochs=15, validation_split=0.1)

In [ ]:
# Inspect sample prediction
sample_img = X[1]
sample_img = sample_img[None,:]
prediction = unet_model.predict(np.float32(sample_img))
prediction = prediction[0]
prediction_mask = np.argmax(prediction, axis=2)
print(prediction_mask)

In [ ]:
# Show truth and sample prediction 
mask = np.argmax(transformed_y[1],axis=2)
print(mask.shape)
plt.imshow(mask, cmap="Paired")
plt.show()

plt.imshow(prediction_mask, cmap="Paired")
plt.show()

In [ ]:
# # Save model and clear session and empty RAM to save resources
unet_model.save("unet.h5")
del unet_model
K.clear_session()
gc.collect()

## Evaluation

<h4> Which metrics should a segmentation task be evaluated? </h4>

<list>
    <li>Pixel-wise accuracy</li>
    <li>IoU</li>
</list>

<h3>Pixel-wise accuracy</h3>

Percentage of pixel classified correctly in the image <br>

<list>
    <li>Easy to understand</li>
    <li> Class imbalance can be a signficiant problem </li>
</list>

![alt text](https://miro.medium.com/max/1400/0*AmruarcqPbBG4jUx.png)


Image source: (Tiu, 2019)
<br>

<h3>IoU</h3>
Intersection over union also known as Jaccard Index measure the overlap of the predicted segmentation and the ground truth over the by the area of the predicted segmentation and the ground truth.
<br>
<br>
<center>
$IoU = \frac{target \cap prediction}{target \cup prediction}$
</center>

<br>

![alt text](https://miro.medium.com/max/600/0*kraYHnYpoJOhaMzq.png)

<br>

For calculating the IoU, the IoU is calculated per class and than averaged

Image source: (Tiu, 2019)

For evaluating the performance of image segmentation models, there are multiple possibilities. Two of them are pixel-wise accuracy and IoU. While pixel-wise accuracy is easy to understand as it is the mean accuracy of the predicted pixels per class, the major disadvantage of this evaluation method is that the scores are effected by class imbalances. As an example, for segmenting ships in the image above, 95% of the pixel does not belong to the ship class. Thus an accuracy of 95% in the ship class would mean that actually, no ship is segmented. For a more realistic evaluation, IoU is used. This concept is known from our last lesson about object detection. The area of the overlap of prediction and target is compared to the area of union of target and prediction. Scores of 0 would mean that the area is not classified correctly at all. The IoU is calculated per class and can then be averaged to give an overall score for the image.

In [ ]:
del X,y, transformed_y 
gc.collect()

In [ ]:
X, y = read_data(name_set=["dataC"])

In [ ]:
X=np.array(X)
y=np.array(y)

In [ ]:
X=X[:100,]
y=y[:100,]

In [ ]:
transformed_y = transform_mask(y=y)

In [ ]:
del y
gc.collect()

In [ ]:
# Predictions for the different models for the test set
model = models.load_model('fcn.h5')
prediction_fcn = np.array(model.predict(X), dtype=np.float16)
prediction_fcn_mask=np.array(np.argmax(prediction_fcn, axis=3), dtype=np.uint8)
del model
K.clear_session()
gc.collect()

In [ ]:
unet_model = models.load_model('unet.h5')
prediction_unet = np.array(unet_model.predict(X), dtype=np.float16)
prediction_unet_mask=np.array(np.argmax(prediction_unet, axis=3), dtype=np.uint8)
del unet_model
K.clear_session()
gc.collect()

In [ ]:
fcn16 = models.load_model("fcn16.h5")
prediction_fcn16 = np.array(fcn16.predict(X), dtype=np.float16)
prediction_fcn16_mask=np.array(np.argmax(prediction_fcn16, axis=3), dtype=np.uint8)
del fcn16
K.clear_session()
gc.collect()

In [ ]:
print(prediction_fcn16_mask.shape)

In [ ]:
# Inspect one sample prediction
# Order: Truth, fcn, fcn-8, u-net
i=11
mask = np.array(np.argmax(transformed_y, axis=3), dtype=np.uint8)

plt.imshow(mask[i], cmap="Paired")
plt.show()

plt.imshow(prediction_fcn_mask[i], cmap="Paired")
plt.show()

plt.imshow(prediction_fcn16_mask[i], cmap="Paired")
plt.show()

plt.imshow(prediction_unet_mask[i], cmap="Paired")
plt.show()

In [ ]:
# Transform predictions to masked predictions (1-channel)
# axis=3 as dimensions: batch, height, width, channels -> maximum per channel

In [ ]:
# Evaluate on accuracy 

def pixel_accuracy(gt, pred, classes):
    r=[]
    
    for c in range(0, classes):
        curr_eval_mask = pred[:, :, c]
        curr_gt_mask = gt[:, :, c]
        
        
        sum_n_ii = np.sum(np.logical_and(curr_eval_mask, curr_gt_mask))
        sum_t_i  = np.sum(curr_gt_mask)
        
        if (sum_t_i == 0):
            pixel_accuracy_ = 0
        else:
            pixel_accuracy_ = sum_n_ii / sum_t_i  

        r.append(pixel_accuracy_)
        
    return r, np.mean(np.array(r), axis=0)


def evaluate_model_pixel_accuracy(ground_truth, prediction, classes):
    results = []
    per_class_results = []
    for i in range(0, len(ground_truth)):
        per_class, mean_accuracy = pixel_accuracy(ground_truth[i,],prediction[i,], classes)

        results.append(mean_accuracy)
        per_class_results.append(per_class)
        
    return per_class_results, np.mean(np.array(results), axis=0)

# Source https://github.com/martinkersner/py_img_seg_eval/blob/master/eval_segm.py

In [ ]:
print("FCN Mean Accuracy")
c, m = evaluate_model_pixel_accuracy(transformed_y, prediction_fcn, classes=num_classes)
print(m)

print("FCN-16 Mean Accuracy")
c, m = evaluate_model_pixel_accuracy(transformed_y, prediction_fcn16, classes=num_classes)
print(m)

print("U-Net Mean Accuracy")
c, m = evaluate_model_pixel_accuracy(transformed_y, prediction_unet, classes=num_classes)
print(m)

In [ ]:
EPS = 1e-12
def get_iou(gt, pr, classes):
    class_wise = np.zeros(classes)
    for cl in range(classes):
        intersection = np.sum((gt == cl)*(pr == cl))
        union = np.sum(np.maximum((gt == cl), (pr == cl)))
        iou = float(intersection)/(union + EPS)
        class_wise[cl] = iou
    return class_wise
# Source https://github.com/divamgupta/image-segmentation-keras/blob/master/keras_segmentation/metrics.py

In [ ]:
def evaluate_model(y, prediction, classes):
    result = []
    for i in range(0, len(y)):
        temp = get_iou(y[i,:], prediction[i,:], classes)
        result.append(temp)
    return np.mean(result, axis=0)

In [ ]:
# Evalute on IoU per Class
print("IoU FCN")
print(evaluate_model(y=transformed_y, prediction=prediction_fcn, classes=num_classes))

print("IoU FCN-16")
print(evaluate_model(y=transformed_y, prediction=prediction_fcn16, classes=num_classes))

print("IoU UNet")
print(evaluate_model(y=transformed_y, prediction=prediction_unet, classes=num_classes))


<h3>References</h3>
<list>
        <li>Dumoulin, V., & Visin, F. (2016). A guide to convolution arithmetic for deep learning. arXiv preprint 
            arXiv:1603.07285.</li>
        <li> Jorden, J. (2018). Evaluating image segmentation models. Retrieved from: 
            https://www.jeremyjordan.me/evaluating-image-segmentation-models/ </li>
            <li> Lane, T. (2018). Transposed convolutions with MS Excel. Retrieved from: https://medium.com/apache-mxnet/transposed-convolutions-explained-with-ms-excel-52d13030c7e8 
        <li>Long, J., Shelhamer, E., & Darrell, T. (2015). Fully convolutional networks for semantic segmentation. 
            In Proceedings of the IEEE conference on computer vision and pattern recognition (pp. 3431-3440).</li>
        <li>Raschka, S., & Mirjalili, V. (2019). Python Machine Learning: Machine Learning and Deep Learning with 
            Python, scikit-learn, and TensorFlow 2. Packt Publishing Ltd.</li>
        <li>Ronneberger, O., Fischer, P., & Brox, T. (2015, October). U-net: Convolutional networks for biomedical 
            image segmentation. In International Conference on Medical image computing and computer-assisted 
            intervention (pp. 234-241). Springer, Cham.</li>
        <li> Tiu, E. (2019). Metrics to evaluate your semantic segmentation model. Retrieved from: 
                https://towardsdatascience.com/metrics-to-evaluate-your-semantic-segmentation-model-6bcb99639aa2 
        </li>
        <li>Vasilev, I. (2019). Advanced Deep Learning With Python: design and implement advanced next-generation 
            ai solutions using tensorflow and pytorch. S.l.: PACKT PUBLISHING LIMITED.</li>
        
</list>